In [0]:
# Cell 1: Audit existing SupplySage Gold tables for UI readiness
# Purpose:
# - Confirm which existing backend tables can directly support the Next.js UI
# - Identify only the missing thin adapter contracts
# Serverless-safe: no cache(), persist(), CACHE TABLE, or restricted Spark configs

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, IntegerType

GOLD_SCHEMA = "supplysage_gold"

expected_ui_backend_tables = [
    {
        "route": "/command-center",
        "purpose": "Supplier risk dashboard cards",
        "table_name": f"{GOLD_SCHEMA}.gold_dashboard_supplier_risk_summary",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "supplier_risk_score",
            "risk_band",
            "active_event_count_30d",
            "impacted_sku_count",
            "final_top_risk_driver",
            "final_recommended_action",
        ],
    },
    {
        "route": "/command-center",
        "purpose": "SKU stockout dashboard cards",
        "table_name": f"{GOLD_SCHEMA}.gold_dashboard_sku_stockout_summary",
        "required_columns": [
            "canonical_sku_id",
            "sku_stockout_risk_score",
            "stockout_risk_band",
            "days_of_cover",
            "revenue_at_risk",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 header/base profile",
        "table_name": f"{GOLD_SCHEMA}.gold_dim_suppliers",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "country",
            "region",
            "category",
            "criticality_tier",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 score and drivers",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_risk_scores",
        "required_columns": [
            "supplier_id",
            "supplier_risk_score",
            "risk_band",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 risk explanation",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_risk_explanation_log",
        "required_columns": [
            "supplier_id",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 SKU dependency graph",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_sku_dependency_mart",
        "required_columns": [
            "supplier_id",
            "canonical_sku_id",
            "is_primary_supplier",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 performance tab",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_performance_mart",
        "required_columns": [
            "supplier_id",
        ],
    },
    {
        "route": "/suppliers/[supplierId], /command-center",
        "purpose": "External risk events and evidence feed",
        "table_name": f"{GOLD_SCHEMA}.gold_external_risk_event_mart",
        "required_columns": [
            "external_event_id",
            "source_name",
            "event_title",
            "event_date",
            "severity",
        ],
    },
    {
        "route": "/suppliers/[supplierId], /alerts/[alertId]",
        "purpose": "RAG evidence cards",
        "table_name": f"{GOLD_SCHEMA}.gold_rag_evidence_chunks",
        "required_columns": [
            "chunk_id",
            "supplier_id",
            "source_name",
            "event_date",
            "severity",
        ],
    },
    {
        "route": "/alerts",
        "purpose": "Operational alert inbox",
        "table_name": f"{GOLD_SCHEMA}.gold_alert_inbox",
        "required_columns": [
            "alert_id",
            "severity",
            "status",
            "entity_type",
            "entity_id",
            "entity_name",
        ],
    },
    {
        "route": "/alerts/[alertId]",
        "purpose": "Alert delivery and email preview status",
        "table_name": f"{GOLD_SCHEMA}.gold_alert_delivery_log",
        "required_columns": [
            "delivery_id",
            "alert_id",
            "recipient_email",
            "channel",
            "delivery_status",
        ],
    },
    {
        "route": "/command-center, /alerts",
        "purpose": "Alert summary breakdown for UI cards",
        "table_name": f"{GOLD_SCHEMA}.gold_alerting_ui_breakdown",
        "required_columns": [
            "breakdown_type",
            "breakdown_key",
            "breakdown_label",
            "metric_name",
            "metric_value",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "Preassembled supplier chat context",
        "table_name": f"{GOLD_SCHEMA}.gold_chat_context_snapshots",
        "required_columns": [
            "supplier_id",
            "supplier_name",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "Agent/LLM run logs",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_risk_agent_run_logs",
        "required_columns": [
            "run_id",
            "supplier_id",
            "question",
            "answer",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "RAG retrieval index",
        "table_name": f"{GOLD_SCHEMA}.gold_rag_retrieval_index",
        "required_columns": [
            "chunk_id",
            "supplier_id",
        ],
    },
]

def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False

audit_rows = []

for item in expected_ui_backend_tables:
    table_name = item["table_name"]
    exists = table_exists(table_name)

    if exists:
        df = spark.table(table_name)
        cols = df.columns
        missing_cols = [c for c in item["required_columns"] if c not in cols]

        # Avoid full expensive counts except for known small/frontend tables.
        if table_name.endswith((
            "gold_alert_inbox",
            "gold_alert_delivery_log",
            "gold_alerting_ui_breakdown",
            "gold_chat_context_snapshots",
            "gold_supplier_risk_agent_run_logs",
        )):
            try:
                row_count = df.count()
            except Exception:
                row_count = None
        else:
            row_count = None

        status = "READY" if len(missing_cols) == 0 else "NEEDS_ADAPTER_OR_PATCH"
    else:
        cols = []
        missing_cols = item["required_columns"]
        row_count = None
        status = "MISSING"

    audit_rows.append({
        "route": item["route"],
        "purpose": item["purpose"],
        "table_name": table_name,
        "exists": exists,
        "status": status,
        "missing_columns": ", ".join(missing_cols),
        "column_count": len(cols),
        "sample_row_count": row_count,
    })

audit_df = spark.createDataFrame(audit_rows)

display(
    audit_df
    .orderBy(
        F.when(F.col("status") == "MISSING", 0)
         .when(F.col("status") == "NEEDS_ADAPTER_OR_PATCH", 1)
         .otherwise(2),
        "route",
        "table_name",
    )
)

ready_count = audit_df.filter(F.col("status") == "READY").count()
patch_count = audit_df.filter(F.col("status") == "NEEDS_ADAPTER_OR_PATCH").count()
missing_count = audit_df.filter(F.col("status") == "MISSING").count()

print("UI backend readiness summary")
print(f"READY: {ready_count}")
print(f"NEEDS_ADAPTER_OR_PATCH: {patch_count}")
print(f"MISSING: {missing_count}")

if missing_count == 0:
    print("Core backend tables exist. Next step should be frontend adapter views/contracts, not rebuilding Gold.")
else:
    print("Some expected backend tables are missing. Patch only the missing route contracts.")

In [0]:
# Cell 2: Inspect only the UI contracts that need adapter/patch work
# Purpose:
# - Show the 6 non-ready tables from the audit
# - Print their actual columns so we can map them into frontend-ready views
# Serverless-safe

from pyspark.sql import functions as F

# Reuse audit_df from Cell 1
needs_adapter_df = (
    audit_df
    .filter(F.col("status") == "NEEDS_ADAPTER_OR_PATCH")
    .select(
        "route",
        "purpose",
        "table_name",
        "missing_columns",
        "column_count",
        "sample_row_count",
    )
    .orderBy("route", "purpose")
)

display(needs_adapter_df)

needs_adapter_rows = needs_adapter_df.collect()

print("Detailed schema inspection for NEEDS_ADAPTER_OR_PATCH tables")
print("=" * 90)

for row in needs_adapter_rows:
    table_name = row["table_name"]
    print(f"\nTABLE: {table_name}")
    print(f"ROUTE: {row['route']}")
    print(f"PURPOSE: {row['purpose']}")
    print(f"MISSING EXPECTED COLUMNS: {row['missing_columns']}")
    print("-" * 90)

    df = spark.table(table_name)

    print("Actual columns:")
    for c in df.columns:
        print(f" - {c}")

    print("\nSample rows:")
    display(df.limit(5))

In [0]:
# Cell 3: Create thin UI adapter views for frontend-friendly contracts
# Purpose:
# - Preserve existing Gold tables as source of truth
# - Add frontend-friendly aliases required by Next.js UI contracts
# - Avoid rebuilding Gold logic
# Serverless-safe: CREATE OR REPLACE VIEW only, no cache/persist/count-heavy logic

GOLD_SCHEMA = "supplysage_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

# -------------------------------------------------------------------
# 1. Command Center supplier risk cards
# Source has:
# - overall_risk_score
# - top_risk_driver
# - recommended_action
# UI wants:
# - supplier_risk_score
# - final_top_risk_driver
# - final_recommended_action
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_command_center_supplier_risk_summary AS
SELECT
    *,
    overall_risk_score AS supplier_risk_score,
    top_risk_driver AS final_top_risk_driver,
    recommended_action AS final_recommended_action
FROM {GOLD_SCHEMA}.gold_dashboard_supplier_risk_summary
""")


# -------------------------------------------------------------------
# 2. Command Center SKU stockout cards
# Source has:
# - stockout_probability
# - sales_exposure_30d
# UI wants:
# - sku_stockout_risk_score
# - revenue_at_risk
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_command_center_sku_stockout_summary AS
SELECT
    *,
    CASE
        WHEN stockout_probability IS NULL THEN NULL
        WHEN stockout_probability <= 1 THEN ROUND(stockout_probability * 100, 2)
        ELSE ROUND(stockout_probability, 2)
    END AS sku_stockout_risk_score,
    sales_exposure_30d AS revenue_at_risk
FROM {GOLD_SCHEMA}.gold_dashboard_sku_stockout_summary
""")


# -------------------------------------------------------------------
# 3. Supplier 360 base profile
# Source has:
# - supplier_category
# UI wants:
# - category
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_dim_suppliers AS
SELECT
    *,
    supplier_category AS category
FROM {GOLD_SCHEMA}.gold_dim_suppliers
""")


# -------------------------------------------------------------------
# 4. Supplier 360 risk score and drivers
# Source has:
# - overall_risk_score
# UI wants:
# - supplier_risk_score
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_supplier_risk_scores AS
SELECT
    *,
    overall_risk_score AS supplier_risk_score
FROM {GOLD_SCHEMA}.gold_supplier_risk_scores
""")


# -------------------------------------------------------------------
# 5. Agent / LLM run logs
# Source has:
# - agent_run_id
# - final_answer
# UI/API wants:
# - run_id
# - answer
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_supplier_risk_agent_run_logs AS
SELECT
    *,
    agent_run_id AS run_id,
    final_answer AS answer
FROM {GOLD_SCHEMA}.gold_supplier_risk_agent_run_logs
""")


# -------------------------------------------------------------------
# 6. Validate adapter views exist and expose required UI fields
# -------------------------------------------------------------------

adapter_expectations = [
    {
        "view_name": f"{GOLD_SCHEMA}.ui_command_center_supplier_risk_summary",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "supplier_risk_score",
            "risk_band",
            "active_event_count_30d",
            "impacted_sku_count",
            "final_top_risk_driver",
            "final_recommended_action",
        ],
    },
    {
        "view_name": f"{GOLD_SCHEMA}.ui_command_center_sku_stockout_summary",
        "required_columns": [
            "canonical_sku_id",
            "sku_stockout_risk_score",
            "stockout_risk_band",
            "days_of_cover",
            "revenue_at_risk",
        ],
    },
    {
        "view_name": f"{GOLD_SCHEMA}.ui_dim_suppliers",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "country",
            "region",
            "category",
            "criticality_tier",
        ],
    },
    {
        "view_name": f"{GOLD_SCHEMA}.ui_supplier_risk_scores",
        "required_columns": [
            "supplier_id",
            "supplier_risk_score",
            "risk_band",
            "top_risk_driver",
            "recommended_action",
        ],
    },
    {
        "view_name": f"{GOLD_SCHEMA}.ui_supplier_risk_agent_run_logs",
        "required_columns": [
            "run_id",
            "supplier_id",
            "question",
            "answer",
            "ok",
            "recommended_action",
            "evidence_count",
        ],
    },
]

print("Validating UI adapter views...")
print("=" * 90)

for item in adapter_expectations:
    view_name = item["view_name"]
    df = spark.table(view_name)
    cols = df.columns
    missing_cols = [c for c in item["required_columns"] if c not in cols]

    print(f"\nVIEW: {view_name}")
    print(f"Column count: {len(cols)}")

    if missing_cols:
        print(f"STATUS: FAILED")
        print(f"Missing columns: {missing_cols}")
        raise ValueError(f"{view_name} is missing required UI columns: {missing_cols}")
    else:
        print("STATUS: READY")

print("\nAll thin UI adapter views are ready.")
print("Next step: rerun the readiness audit using these ui_* views as frontend contracts.")

In [0]:
# Cell 4: Rerun UI backend readiness audit using frontend adapter views
# Purpose:
# - Confirm frontend-facing ui_* contracts are ready
# - Keep raw Gold marts as source-of-truth, but route Next.js API/data adapters to ui_* views
# Serverless-safe

from pyspark.sql import functions as F

GOLD_SCHEMA = "supplysage_gold"

frontend_contract_tables = [
    {
        "route": "/command-center",
        "purpose": "Supplier risk dashboard cards",
        "table_name": f"{GOLD_SCHEMA}.ui_command_center_supplier_risk_summary",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "supplier_risk_score",
            "risk_band",
            "active_event_count_30d",
            "impacted_sku_count",
            "final_top_risk_driver",
            "final_recommended_action",
        ],
    },
    {
        "route": "/command-center",
        "purpose": "SKU stockout dashboard cards",
        "table_name": f"{GOLD_SCHEMA}.ui_command_center_sku_stockout_summary",
        "required_columns": [
            "canonical_sku_id",
            "sku_stockout_risk_score",
            "stockout_risk_band",
            "days_of_cover",
            "revenue_at_risk",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 header/base profile",
        "table_name": f"{GOLD_SCHEMA}.ui_dim_suppliers",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "country",
            "region",
            "category",
            "criticality_tier",
            "annual_spend",
            "single_source_flag",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 score and drivers",
        "table_name": f"{GOLD_SCHEMA}.ui_supplier_risk_scores",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "supplier_risk_score",
            "risk_band",
            "top_risk_driver",
            "recommended_action",
            "operational_score",
            "dependency_score",
            "external_event_score",
            "logistics_score",
            "sanctions_score",
            "cyber_score",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 risk explanation",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_risk_explanation_log",
        "required_columns": [
            "supplier_id",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 SKU dependency graph",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_sku_dependency_mart",
        "required_columns": [
            "supplier_id",
            "canonical_sku_id",
            "is_primary_supplier",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 performance tab",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_performance_mart",
        "required_columns": [
            "supplier_id",
        ],
    },
    {
        "route": "/suppliers/[supplierId], /command-center",
        "purpose": "External risk events and evidence feed",
        "table_name": f"{GOLD_SCHEMA}.gold_external_risk_event_mart",
        "required_columns": [
            "external_event_id",
            "source_name",
            "event_title",
            "event_date",
            "severity",
        ],
    },
    {
        "route": "/suppliers/[supplierId], /alerts/[alertId]",
        "purpose": "RAG evidence cards",
        "table_name": f"{GOLD_SCHEMA}.gold_rag_evidence_chunks",
        "required_columns": [
            "chunk_id",
            "supplier_id",
            "source_name",
            "event_date",
            "severity",
        ],
    },
    {
        "route": "/alerts",
        "purpose": "Operational alert inbox",
        "table_name": f"{GOLD_SCHEMA}.gold_alert_inbox",
        "required_columns": [
            "alert_id",
            "severity",
            "status",
            "entity_type",
            "entity_id",
            "entity_name",
        ],
    },
    {
        "route": "/alerts/[alertId]",
        "purpose": "Alert delivery and email preview status",
        "table_name": f"{GOLD_SCHEMA}.gold_alert_delivery_log",
        "required_columns": [
            "delivery_id",
            "alert_id",
            "recipient_email",
            "channel",
            "delivery_status",
        ],
    },
    {
        "route": "/command-center, /alerts",
        "purpose": "Alert summary breakdown for UI cards",
        "table_name": f"{GOLD_SCHEMA}.gold_alerting_ui_breakdown",
        "required_columns": [
            "breakdown_type",
            "breakdown_key",
            "breakdown_label",
            "metric_name",
            "metric_value",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "Preassembled supplier chat context",
        "table_name": f"{GOLD_SCHEMA}.gold_chat_context_snapshots",
        "required_columns": [
            "supplier_id",
            "supplier_name",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "Agent/LLM run logs",
        "table_name": f"{GOLD_SCHEMA}.ui_supplier_risk_agent_run_logs",
        "required_columns": [
            "run_id",
            "supplier_id",
            "question",
            "answer",
            "ok",
            "recommended_action",
            "evidence_count",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "RAG retrieval index",
        "table_name": f"{GOLD_SCHEMA}.gold_rag_retrieval_index",
        "required_columns": [
            "chunk_id",
            "supplier_id",
        ],
    },
]

def table_or_view_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False

audit_rows = []

for item in frontend_contract_tables:
    table_name = item["table_name"]
    exists = table_or_view_exists(table_name)

    if exists:
        df = spark.table(table_name)
        cols = df.columns
        missing_cols = [c for c in item["required_columns"] if c not in cols]
        status = "READY" if len(missing_cols) == 0 else "NEEDS_ADAPTER_OR_PATCH"
    else:
        cols = []
        missing_cols = item["required_columns"]
        status = "MISSING"

    audit_rows.append({
        "route": item["route"],
        "purpose": item["purpose"],
        "frontend_contract": table_name,
        "exists": exists,
        "status": status,
        "missing_columns": ", ".join(missing_cols),
        "column_count": len(cols),
    })

frontend_audit_df = spark.createDataFrame(audit_rows)

display(
    frontend_audit_df
    .orderBy(
        F.when(F.col("status") == "MISSING", 0)
         .when(F.col("status") == "NEEDS_ADAPTER_OR_PATCH", 1)
         .otherwise(2),
        "route",
        "frontend_contract",
    )
)

ready_count = frontend_audit_df.filter(F.col("status") == "READY").count()
patch_count = frontend_audit_df.filter(F.col("status") == "NEEDS_ADAPTER_OR_PATCH").count()
missing_count = frontend_audit_df.filter(F.col("status") == "MISSING").count()

print("Frontend contract readiness summary")
print(f"READY: {ready_count}")
print(f"NEEDS_ADAPTER_OR_PATCH: {patch_count}")
print(f"MISSING: {missing_count}")

if patch_count == 0 and missing_count == 0:
    print("All core frontend backend contracts are ready.")
    print("Next step: create final UI contract registry table for the Next.js app.")
else:
    print("Some contracts still need adapter work. Inspect rows above before frontend build.")

In [0]:
# Cell 5: Create UI adapter view for /alerts operational inbox
# Purpose:
# - Keep gold_alert_inbox as source of truth
# - Add frontend-required aliases:
#   status, entity_type, entity_id, entity_name
# - Use safe defaults if the raw table does not contain workflow fields yet
# Serverless-safe: CREATE OR REPLACE VIEW only

GOLD_SCHEMA = "supplysage_gold"

SOURCE_TABLE = f"{GOLD_SCHEMA}.gold_alert_inbox"
TARGET_VIEW = f"{GOLD_SCHEMA}.ui_alert_inbox"

source_df = spark.table(SOURCE_TABLE)
source_cols = source_df.columns
source_col_set = set(source_cols)

print(f"Source table: {SOURCE_TABLE}")
print("Source columns:")
for c in source_cols:
    print(f" - {c}")

def q(col_name: str) -> str:
    return f"`{col_name}`"

def first_existing(candidates):
    for c in candidates:
        if c in source_col_set:
            return c
    return None

# -------------------------------------------------------------------
# status
# UI wants operational workflow statuses:
# New, Acknowledged, Investigating, Mitigated, Resolved
# If no workflow status exists yet, default current alerts to New.
# -------------------------------------------------------------------

status_source = first_existing([
    "status",
    "alert_status",
    "workflow_status",
    "inbox_status",
    "case_status",
])

if status_source:
    status_expr = f"""
        CASE
            WHEN {q(status_source)} IS NULL THEN 'New'
            WHEN lower({q(status_source)}) IN ('new', 'open', 'created', 'drafted') THEN 'New'
            WHEN lower({q(status_source)}) IN ('acknowledged', 'ack') THEN 'Acknowledged'
            WHEN lower({q(status_source)}) IN ('investigating', 'in_progress', 'in progress') THEN 'Investigating'
            WHEN lower({q(status_source)}) IN ('mitigated') THEN 'Mitigated'
            WHEN lower({q(status_source)}) IN ('resolved', 'closed') THEN 'Resolved'
            ELSE initcap({q(status_source)})
        END
    """
else:
    status_expr = "'New'"

# -------------------------------------------------------------------
# entity_type
# Prefer explicit source if present.
# Otherwise infer supplier vs sku from available identifiers.
# -------------------------------------------------------------------

entity_type_source = first_existing([
    "entity_type",
    "alert_entity_type",
    "risk_entity_type",
    "target_entity_type",
])

supplier_id_source = first_existing([
    "supplier_id",
    "alert_supplier_id",
    "impacted_supplier_id",
])

sku_id_source = first_existing([
    "canonical_sku_id",
    "sku_id",
    "retail_product_id",
    "m5_item_id",
    "alert_sku_id",
])

if entity_type_source:
    entity_type_expr = f"""
        CASE
            WHEN lower({q(entity_type_source)}) IN ('supplier', 'suppliers') THEN 'supplier'
            WHEN lower({q(entity_type_source)}) IN ('sku', 'skus', 'product', 'products') THEN 'sku'
            ELSE lower({q(entity_type_source)})
        END
    """
elif supplier_id_source and sku_id_source:
    entity_type_expr = f"""
        CASE
            WHEN {q(supplier_id_source)} IS NOT NULL THEN 'supplier'
            WHEN {q(sku_id_source)} IS NOT NULL THEN 'sku'
            ELSE 'supplier'
        END
    """
elif supplier_id_source:
    entity_type_expr = "'supplier'"
elif sku_id_source:
    entity_type_expr = "'sku'"
else:
    entity_type_expr = "'supplier'"

# -------------------------------------------------------------------
# entity_id
# Prefer explicit entity id.
# Otherwise choose supplier_id first, then sku_id.
# -------------------------------------------------------------------

entity_id_source = first_existing([
    "entity_id",
    "alert_entity_id",
    "risk_entity_id",
    "target_entity_id",
])

if entity_id_source:
    entity_id_expr = f"CAST({q(entity_id_source)} AS STRING)"
elif supplier_id_source and sku_id_source:
    entity_id_expr = f"CAST(coalesce({q(supplier_id_source)}, {q(sku_id_source)}) AS STRING)"
elif supplier_id_source:
    entity_id_expr = f"CAST({q(supplier_id_source)} AS STRING)"
elif sku_id_source:
    entity_id_expr = f"CAST({q(sku_id_source)} AS STRING)"
else:
    entity_id_expr = "CAST(alert_id AS STRING)"

# -------------------------------------------------------------------
# entity_name
# Prefer explicit entity name.
# Otherwise choose supplier_name first, then SKU/product fields.
# -------------------------------------------------------------------

entity_name_source = first_existing([
    "entity_name",
    "alert_entity_name",
    "risk_entity_name",
    "target_entity_name",
])

supplier_name_source = first_existing([
    "supplier_name",
    "alert_supplier_name",
    "impacted_supplier_name",
])

sku_name_source = first_existing([
    "sku_name",
    "product_name",
    "canonical_sku_name",
    "retail_product_name",
])

if entity_name_source:
    entity_name_expr = f"CAST({q(entity_name_source)} AS STRING)"
elif supplier_name_source and sku_name_source:
    entity_name_expr = f"CAST(coalesce({q(supplier_name_source)}, {q(sku_name_source)}) AS STRING)"
elif supplier_name_source:
    entity_name_expr = f"CAST({q(supplier_name_source)} AS STRING)"
elif sku_name_source:
    entity_name_expr = f"CAST({q(sku_name_source)} AS STRING)"
else:
    entity_name_expr = entity_id_expr

# -------------------------------------------------------------------
# Create frontend adapter view
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {TARGET_VIEW} AS
SELECT
    *,
    {status_expr} AS status,
    {entity_type_expr} AS entity_type,
    {entity_id_expr} AS entity_id,
    {entity_name_expr} AS entity_name
FROM {SOURCE_TABLE}
""")

# -------------------------------------------------------------------
# Validate adapter view
# -------------------------------------------------------------------

required_cols = [
    "alert_id",
    "severity",
    "status",
    "entity_type",
    "entity_id",
    "entity_name",
]

target_df = spark.table(TARGET_VIEW)
target_cols = target_df.columns
missing_cols = [c for c in required_cols if c not in target_cols]

print(f"\nCreated adapter view: {TARGET_VIEW}")
print("Required column check:")

if missing_cols:
    print(f"FAILED. Missing: {missing_cols}")
    raise ValueError(f"{TARGET_VIEW} is missing required columns: {missing_cols}")

print("READY")
print("Adapter columns added:")
print(" - status")
print(" - entity_type")
print(" - entity_id")
print(" - entity_name")

display(
    target_df
    .select(*required_cols)
    .limit(10)
)

In [0]:
# Cell 6: Final UI backend readiness audit using all frontend-facing ui_* contracts
# Purpose:
# - Confirm all core frontend contracts are READY
# - Use ui_alert_inbox instead of raw gold_alert_inbox
# - This is the final audit before building the Next.js UI
# Serverless-safe

from pyspark.sql import functions as F

GOLD_SCHEMA = "supplysage_gold"

final_frontend_contract_tables = [
    {
        "route": "/command-center",
        "purpose": "Supplier risk dashboard cards",
        "frontend_contract": f"{GOLD_SCHEMA}.ui_command_center_supplier_risk_summary",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "supplier_risk_score",
            "risk_band",
            "active_event_count_30d",
            "impacted_sku_count",
            "final_top_risk_driver",
            "final_recommended_action",
        ],
    },
    {
        "route": "/command-center",
        "purpose": "SKU stockout dashboard cards",
        "frontend_contract": f"{GOLD_SCHEMA}.ui_command_center_sku_stockout_summary",
        "required_columns": [
            "canonical_sku_id",
            "sku_stockout_risk_score",
            "stockout_risk_band",
            "days_of_cover",
            "revenue_at_risk",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 header/base profile",
        "frontend_contract": f"{GOLD_SCHEMA}.ui_dim_suppliers",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "country",
            "region",
            "category",
            "criticality_tier",
            "annual_spend",
            "single_source_flag",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 score and drivers",
        "frontend_contract": f"{GOLD_SCHEMA}.ui_supplier_risk_scores",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "supplier_risk_score",
            "risk_band",
            "top_risk_driver",
            "recommended_action",
            "operational_score",
            "dependency_score",
            "external_event_score",
            "logistics_score",
            "sanctions_score",
            "cyber_score",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 risk explanation",
        "frontend_contract": f"{GOLD_SCHEMA}.gold_supplier_risk_explanation_log",
        "required_columns": [
            "supplier_id",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 SKU dependency graph",
        "frontend_contract": f"{GOLD_SCHEMA}.gold_supplier_sku_dependency_mart",
        "required_columns": [
            "supplier_id",
            "canonical_sku_id",
            "is_primary_supplier",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 performance tab",
        "frontend_contract": f"{GOLD_SCHEMA}.gold_supplier_performance_mart",
        "required_columns": [
            "supplier_id",
        ],
    },
    {
        "route": "/suppliers/[supplierId], /command-center",
        "purpose": "External risk events and evidence feed",
        "frontend_contract": f"{GOLD_SCHEMA}.gold_external_risk_event_mart",
        "required_columns": [
            "external_event_id",
            "source_name",
            "event_title",
            "event_date",
            "severity",
        ],
    },
    {
        "route": "/suppliers/[supplierId], /alerts/[alertId]",
        "purpose": "RAG evidence cards",
        "frontend_contract": f"{GOLD_SCHEMA}.gold_rag_evidence_chunks",
        "required_columns": [
            "chunk_id",
            "supplier_id",
            "source_name",
            "event_date",
            "severity",
        ],
    },
    {
        "route": "/alerts",
        "purpose": "Operational alert inbox",
        "frontend_contract": f"{GOLD_SCHEMA}.ui_alert_inbox",
        "required_columns": [
            "alert_id",
            "severity",
            "status",
            "entity_type",
            "entity_id",
            "entity_name",
        ],
    },
    {
        "route": "/alerts/[alertId]",
        "purpose": "Alert delivery and email preview status",
        "frontend_contract": f"{GOLD_SCHEMA}.gold_alert_delivery_log",
        "required_columns": [
            "delivery_id",
            "alert_id",
            "recipient_email",
            "channel",
            "delivery_status",
        ],
    },
    {
        "route": "/command-center, /alerts",
        "purpose": "Alert summary breakdown for UI cards",
        "frontend_contract": f"{GOLD_SCHEMA}.gold_alerting_ui_breakdown",
        "required_columns": [
            "breakdown_type",
            "breakdown_key",
            "breakdown_label",
            "metric_name",
            "metric_value",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "Preassembled supplier chat context",
        "frontend_contract": f"{GOLD_SCHEMA}.gold_chat_context_snapshots",
        "required_columns": [
            "supplier_id",
            "supplier_name",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "Agent/LLM run logs",
        "frontend_contract": f"{GOLD_SCHEMA}.ui_supplier_risk_agent_run_logs",
        "required_columns": [
            "run_id",
            "supplier_id",
            "question",
            "answer",
            "ok",
            "recommended_action",
            "evidence_count",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "RAG retrieval index",
        "frontend_contract": f"{GOLD_SCHEMA}.gold_rag_retrieval_index",
        "required_columns": [
            "chunk_id",
            "supplier_id",
        ],
    },
]

def table_or_view_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False

audit_rows = []

for item in final_frontend_contract_tables:
    contract_name = item["frontend_contract"]
    exists = table_or_view_exists(contract_name)

    if exists:
        df = spark.table(contract_name)
        cols = df.columns
        missing_cols = [c for c in item["required_columns"] if c not in cols]
        status = "READY" if len(missing_cols) == 0 else "NEEDS_ADAPTER_OR_PATCH"
    else:
        cols = []
        missing_cols = item["required_columns"]
        status = "MISSING"

    audit_rows.append({
        "route": item["route"],
        "purpose": item["purpose"],
        "frontend_contract": contract_name,
        "exists": exists,
        "status": status,
        "missing_columns": ", ".join(missing_cols),
        "column_count": len(cols),
    })

final_ui_audit_df = spark.createDataFrame(audit_rows)

display(
    final_ui_audit_df
    .orderBy(
        F.when(F.col("status") == "MISSING", 0)
         .when(F.col("status") == "NEEDS_ADAPTER_OR_PATCH", 1)
         .otherwise(2),
        "route",
        "frontend_contract",
    )
)

ready_count = final_ui_audit_df.filter(F.col("status") == "READY").count()
patch_count = final_ui_audit_df.filter(F.col("status") == "NEEDS_ADAPTER_OR_PATCH").count()
missing_count = final_ui_audit_df.filter(F.col("status") == "MISSING").count()

print("Final frontend contract readiness summary")
print(f"READY: {ready_count}")
print(f"NEEDS_ADAPTER_OR_PATCH: {patch_count}")
print(f"MISSING: {missing_count}")

if patch_count == 0 and missing_count == 0:
    print("✅ All core UI backend contracts are ready.")
    print("Use the frontend_contract values above as the Databricks SQL sources for the Next.js data adapter/API layer.")
else:
    print("⚠️ Some contracts still need adapter work. Inspect rows above.")